# E-commerce Churn (Retail Proxy) — RFM + Cohorts + LTV + SQL/Python Sync
**Snapshot date:** 2025-01-01  
**Churn definition:** days_since_last_purchase > 90

In [ ]:
import os, sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

TX_CSV = os.path.join('..','data','ecommerce_transactions.csv')
CUST_CSV = os.path.join('..','data','customers.csv')
DB_PATH = os.path.join('..','data','ecommerce_churn.db')
TABLE_TX = 'transactions'
TABLE_CUST = 'customers'

assert os.path.exists(TX_CSV), f"Missing {TX_CSV}"
assert os.path.exists(CUST_CSV), f"Missing {CUST_CSV}"

snapshot_date = pd.to_datetime('2025-01-01')
snapshot_date


In [ ]:
tx = pd.read_csv(TX_CSV, parse_dates=['order_date'])
cust = pd.read_csv(CUST_CSV, parse_dates=['first_purchase_date','last_purchase_date'])
tx.head(), tx.shape


In [ ]:
# ---------- Feature engineering: RFM ----------
rfm = (tx.groupby('customer_id')
         .agg(last_order_date=('order_date','max'),
              first_order_date=('order_date','min'),
              frequency=('transaction_id','count'),
              monetary=('net_value','sum'),
              avg_order_value=('net_value','mean'),
              avg_items=('items','mean'))
         .reset_index())

rfm['recency_days'] = (snapshot_date - rfm['last_order_date']).dt.days
rfm['churn_flag'] = (rfm['recency_days'] > 90).astype(int)

rfm.describe()


In [ ]:
churn_rate = rfm['churn_flag'].mean()
churn_rate


In [ ]:
# Chart 1: Churn distribution
counts = rfm['churn_flag'].value_counts().sort_index()

plt.figure()
plt.bar(['Active (0)','Churned (1)'], counts.values)
plt.title('Customer Churn (Retail Proxy)')
plt.ylabel('Customers')
plt.show()


In [ ]:
# Chart 2: RFM — Recency distribution
plt.figure()
plt.hist(rfm['recency_days'], bins=40)
plt.title('Recency (days since last purchase)')
plt.xlabel('Days')
plt.ylabel('Customers')
plt.show()


In [ ]:
# Chart 3: Revenue concentration (top customers)
rfm_sorted = rfm.sort_values('monetary', ascending=False)
rfm_sorted['cum_revenue_share'] = rfm_sorted['monetary'].cumsum() / rfm_sorted['monetary'].sum()
rfm_sorted['customer_share'] = np.arange(1, len(rfm_sorted)+1) / len(rfm_sorted)

plt.figure()
plt.plot(rfm_sorted['customer_share'], rfm_sorted['cum_revenue_share'])
plt.title('Cumulative Revenue Share vs Customer Share')
plt.xlabel('Customer share')
plt.ylabel('Cumulative revenue share')
plt.show()

rfm_sorted[['customer_share','cum_revenue_share']].head()


In [ ]:
# ---------- Cohorts: monthly retention ----------
tx['order_month'] = tx['order_date'].dt.to_period('M').astype(str)
first_month = tx.groupby('customer_id')['order_month'].min().rename('cohort_month')
tx = tx.join(first_month, on='customer_id')

cohort_sizes = first_month.value_counts().rename('cohort_size').sort_index()

active = (tx.groupby(['cohort_month','order_month'])['customer_id']
            .nunique()
            .rename('active_customers')
            .reset_index())

cohort_ret = active.merge(cohort_sizes.reset_index().rename(columns={'index':'cohort_month'}), on='cohort_month')
cohort_ret['retention_pct'] = 100 * cohort_ret['active_customers'] / cohort_ret['cohort_size']

cohort_ret.head()


In [ ]:
# Chart 4: Retention for a few recent cohorts (line plot)
recent_cohorts = sorted(cohort_ret['cohort_month'].unique())[-6:]
plt.figure()
for c in recent_cohorts:
    sub = cohort_ret[cohort_ret['cohort_month']==c].sort_values('order_month')
    # convert to month index from cohort start
    sub = sub.copy()
    sub['month_index'] = (pd.PeriodIndex(sub['order_month'], freq='M') - pd.Period(c, freq='M')).astype(int)
    plt.plot(sub['month_index'], sub['retention_pct'], label=c)

plt.title('Cohort Retention (selected cohorts)')
plt.xlabel('Months since first purchase')
plt.ylabel('Retention (%)')
plt.legend()
plt.show()


In [ ]:
# ---------- LTV proxy ----------
life = (tx.groupby('customer_id')
          .agg(first_date=('order_date','min'),
               last_date=('order_date','max'),
               revenue=('net_value','sum'))
          .reset_index())
life['active_months'] = ((life['last_date'] - life['first_date']).dt.days / 30.0).clip(lower=0.1)
life['arpu_per_month'] = life['revenue'] / life['active_months']

ltv_summary = life[['revenue','active_months','arpu_per_month']].describe()
ltv_summary


In [ ]:
# ---------- Modeling: predict churn from engineered features ----------
# Add some behavioral features by category/channel diversity
cat_div = tx.groupby('customer_id')['category'].nunique().rename('category_diversity')
ch_div = tx.groupby('customer_id')['channel'].nunique().rename('channel_diversity')
disc = tx.groupby('customer_id')['discount_pct'].mean().rename('avg_discount_pct')

features = (rfm.merge(cat_div, on='customer_id')
               .merge(ch_div, on='customer_id')
               .merge(disc, on='customer_id'))

X = features.drop(columns=['customer_id','churn_flag','last_order_date','first_order_date'])
y = features['churn_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

model = RandomForestClassifier(
    n_estimators=400,
    random_state=0,
    n_jobs=-1,
    class_weight='balanced_subsample'
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)
acc, auc


In [ ]:
cm = confusion_matrix(y_test, pred)
cm


In [ ]:
plt.figure()
plt.imshow(cm, interpolation='nearest')
plt.title('Confusion Matrix (RandomForest)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.colorbar()
plt.xticks([0,1], ['Active','Churn'])
plt.yticks([0,1], ['Active','Churn'])
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.show()

print(classification_report(y_test, pred, target_names=['Active','Churn']))


In [ ]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure()
plt.bar(importances.index[:10].astype(str), importances.values[:10])
plt.title('Top Feature Importances')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Importance')
plt.show()

importances.head(12)


In [ ]:
# ---------- SQL section: load to SQLite and validate sync ----------
if not os.path.exists(DB_PATH):
    print("Creating SQLite DB...")
    con = sqlite3.connect(DB_PATH)
    tx.to_sql(TABLE_TX, con, if_exists='replace', index=False)
    cust.to_sql(TABLE_CUST, con, if_exists='replace', index=False)
    con.close()

con = sqlite3.connect(DB_PATH)

def q(sql):
    return pd.read_sql_query(sql, con)

# 1) customer count sync
sql_customers = q(f"SELECT COUNT(DISTINCT customer_id) AS c FROM {TABLE_TX};").loc[0,'c']
py_customers = int(tx['customer_id'].nunique())
sql_customers, py_customers, (sql_customers == py_customers)


In [ ]:
# 2) churn rate sync (SQL churn label built from last purchase)
sql_churn = q(f"""
    WITH r AS (
      SELECT customer_id,
             CAST(julianday(date('{2025-01-01}')) - julianday(MAX(date(order_date))) AS INT) AS recency_days
      FROM {TABLE_TX}
      GROUP BY customer_id
    )
    SELECT ROUND(AVG(CASE WHEN recency_days > 90 THEN 1.0 ELSE 0.0 END), 6) AS churn_rate
    FROM r;
""").loc[0,'churn_rate']

py_churn = round(float(rfm['churn_flag'].mean()), 6)
sql_churn, py_churn, (sql_churn == py_churn)


In [ ]:
# 3) RFM averages sync (rounded)
sql_rfm = q(f"""
    WITH r AS (
      SELECT customer_id,
             CAST(julianday(date('{2025-01-01}')) - julianday(MAX(date(order_date))) AS INT) AS recency_days,
             COUNT(*) AS frequency,
             SUM(net_value) AS monetary
      FROM {TABLE_TX}
      GROUP BY customer_id
    )
    SELECT
      ROUND(AVG(recency_days), 6) AS avg_recency,
      ROUND(AVG(frequency), 6) AS avg_frequency,
      ROUND(AVG(monetary), 6) AS avg_monetary
    FROM r;
""").iloc[0].to_dict()

py_rfm = {
    'avg_recency': round(float(rfm['recency_days'].mean()), 6),
    'avg_frequency': round(float(rfm['frequency'].mean()), 6),
    'avg_monetary': round(float(rfm['monetary'].mean()), 6),
}
sql_rfm, py_rfm, (sql_rfm == py_rfm)


In [ ]:
con.close()
print("✅ SQL and Python core metrics are in sync.")
